# Chapitre 1.5 : le pré-entraînement

Le pré-entraînement, c'est la longue étape où « ton IA » lit des quantités énormes de texte pour apprendre à prédire le jeton suivant. Dans ce carnet, tu le fais en tout petit : un modèle de quelques millions de nombres, 60 pas, quelques secondes. Tu y vois ce que fait la vraie boucle de forge (le moteur de « ton IA »), et surtout ce qui la rend fiable sur des jours de calcul : **on peut l'arrêter et la reprendre sans rien perdre**.

Ce carnet accompagne [la page du chapitre 1.5](https://chipat-neko.github.io/forge-ia/partie-1/1-5/) : tu peux la lire avant, ou en même temps.

**Ce que tu vas faire**

1. mesurer la perte d'un modèle tout neuf, et la comparer à ln V ;
2. regarder le planning du taux d'apprentissage ;
3. entraîner un petit modèle pendant 60 pas ;
4. l'arrêter à mi-chemin comme avec Ctrl+C, le reprendre, et vérifier qu'il finit sur la même perte ;
5. voir ce qu'il faut pour la version d'une heure de « ton IA » ;
6. relever un défi : retrouver dans un journal le pas où l'entraînement a été mis en pause.

**Comment l'utiliser**

Lance les cellules de code de haut en bas : clique sur le bouton rond avec un triangle, à gauche de la cellule, ou tape Maj + Entrée. Tu peux aussi choisir « Exécution », puis « Tout exécuter ». Tu peux modifier le code et le relancer autant de fois que tu veux : tu ne peux rien casser.

Ce carnet calcule sur la carte graphique si Colab t'en donne une, sinon sur le processeur : il te le dit. Les deux marchent, et l'essai dure quelques secondes à quelques dizaines de secondes.

## Étape 0 : récupérer le code

Le code du projet est dans un dépôt GitHub public. La cellule suivante le télécharge avec `git clone`, installe la petite bibliothèque `regex` dont le tokenizer a besoin, puis indique à Python où trouver le projet et forge. Elle choisit aussi l'appareil de calcul, et définit `nombre`, qui écrit les nombres à la française.

In [ ]:
import os
import sys
from pathlib import Path

# Sur Colab, on télécharge le dépôt public. Quand ce carnet est lancé par les tests
# du projet, la variable FORGE_IA_LOCAL donne le chemin d’un dépôt déjà présent :
# on l’utilise, sans rien télécharger ni installer.
depot_local = os.environ.get("FORGE_IA_LOCAL")
if depot_local:
    depot = Path(depot_local)
else:
    depot = Path("forge-ia")
    if not depot.exists():
        !git clone --depth 1 https://github.com/chipat-neko/forge-ia
    !pip install --quiet regex

projet = depot / "projets" / "1.5-pre-entrainement"
if not projet.exists():
    raise FileNotFoundError(
        "Le dossier projets/1.5-pre-entrainement est introuvable. Si le dépôt forge-ia était déjà là, "
        "supprime-le (panneau des fichiers, à gauche), puis relance cette cellule\u202f; sinon, vérifie ta connexion."
    )

# Python cherche les modules dans les dossiers de sys.path : on y ajoute le projet et forge.
sys.path.insert(0, str(projet.resolve()))
sys.path.insert(0, str((depot / "projets" / "forge").resolve()))

import math
import shutil
import tempfile

import torch
from forge.entrainement import ChargeurJetons, entrainer
from forge.entrainement.planning import taux
from forge.modele import Transformer, nombre_parametres

from pre_entrainement import journal
from pre_entrainement.appareil import choisir_appareil
from pre_entrainement.essai import config_essai, essai_pause_reprise, preparer_essai, texte_du_site

# La carte si PyTorch la voit, sinon le processeur (FORGE_APPAREIL=cpu force le processeur).
appareil = choisir_appareil().type


def nombre(x, decimales=0):
    # À la française : une espace fine entre les milliers, une virgule pour les décimales.
    return f"{x:,.{decimales}f}".replace(",", "\u202f").replace(".", ",")


# Un dossier à nous pour les données, le journal et les sauvegardes de l’essai.
racine = Path(tempfile.mkdtemp(prefix="pre-entrainement-"))
dossier = preparer_essai(racine, texte_du_site())
config = config_essai(appareil, racine)  # 60 pas de 1 024 jetons
lieu = "la carte graphique" if appareil == "cuda" else "le processeur"
print(f"Ce carnet calcule sur {lieu}, avec PyTorch {torch.__version__.split('+')[0]}.")
print(f"Le modèle de l’essai a {nombre(nombre_parametres(config.modele))} paramètres (« mini », la version d’une heure, en a 23\u202f204\u202f736).")

## Étape 1 : la perte au départ

Un modèle neuf n'a rien appris : ses poids sont tirés au hasard. Il doit donner à peu près la même probabilité à chacun des V = 32 768 jetons possibles. La **perte** est `-ln` de la probabilité qu'il donne au vrai jeton suivant : un modèle qui devine au hasard a donc une perte de `ln V`, environ 10,4. Pendant l'entraînement, elle doit descendre.

La cellule prépare un lot de texte (le texte du site, encodé par le tokenizer de « ton IA ») et mesure la perte du modèle neuf.

In [ ]:
torch.manual_seed(config.graine)
modele_neuf = Transformer(config.modele)
chargeur = ChargeurJetons(dossier / "train", config.modele.longueur_contexte, config.micro_lot, config.graine)
x, y = chargeur.lot()  # 16 lignes de 64 jetons, et pour chacune le jeton qui suit chaque position
with torch.no_grad():
    _, perte_neuve = modele_neuf(x, y)

hasard = math.log(config.modele.taille_vocabulaire)
print(f"Jetons à lire\u202f: {nombre(chargeur.jetons)} pour l’entraînement, {nombre(chargeur.morceaux)} morceaux de {config.modele.longueur_contexte + 1} jetons.")
print(f"ln V = ln {nombre(config.modele.taille_vocabulaire)} = {nombre(hasard, 3)}")
print(f"Perte du modèle neuf\u202f: {nombre(perte_neuve.item(), 3)}")
print(f"Elle est proche du hasard\u202f: {abs(perte_neuve.item() - hasard) < 0.1}")
del chargeur, x, y  # on rend le fichier de jetons : l’essai de l’étape 4 le retrouve tel quel

## Étape 2 : le planning du taux d'apprentissage

Le taux d'apprentissage dit de combien on bouge les poids à chaque pas. Au début, les gradients sont désordonnés : de grands pas déstabiliseraient le modèle. On **monte** donc le taux en ligne droite pendant les premiers pas (l'échauffement), puis on le **redescend** en suivant une demi-vague de cosinus, jusqu'à un dixième de sa valeur maximale. C'est la fonction `taux` de forge, la même que pour la version d'une heure (qui a 6 370 pas et 100 pas d'échauffement).

In [ ]:
print(f"{config.pas_total} pas, taux maximal {nombre(config.taux_max, 4)}, échauffement de {config.echauffement} pas")
for pas in (1, 3, config.echauffement, 15, 30, 45, config.pas_total):
    t = taux(pas, config.pas_total, config.taux_max, config.echauffement)
    print(f"  pas {pas:>2}\u202f: taux {nombre(t, 6)}")
print(f"Le taux final est le dixième du maximum\u202f: {math.isclose(taux(config.pas_total, config.pas_total, config.taux_max, config.echauffement), config.taux_max / 10)}")

## Étape 3 : un petit entraînement de 60 pas

La fonction `entrainer` est celle de forge, celle qui lance la version d'une heure : elle prend la configuration et fait les pas, un par un, en écrivant une ligne dans un **journal** à chaque pas. Elle mesure aussi la perte de validation (sur des textes que le modèle n'a jamais lus) tous les 20 pas, et sauvegarde tout à la fin.

Chaque pas lit 1 024 jetons. En 60 pas, le modèle voit 61 440 jetons : quelques lectures du petit texte.

In [ ]:
resultat = entrainer(config)

lignes = journal.lire(config.dossier_sortie / "journal.jsonl")
r = journal.resume(lignes, config)
print()
print(f"Pas faits\u202f: {r['pas_faits']} sur {r['pas_total']}, {nombre(r['jetons_vus'])} jetons vus.")
print(f"Perte du premier pas\u202f: {nombre(r['perte_premier_pas'], 3)} (le hasard\u202f: {nombre(hasard, 3)}).")
print(f"Perte du dernier pas\u202f: {nombre(r['perte_dernier_pas'], 3)}, validation {nombre(r['validation_finale'], 3)}.")
print(f"Une session, fin\u202f: {journal.sessions(lignes)[0]['fin']}.")

## Étape 4 : Ctrl+C, puis la reprise

Pendant une longue session, on veut pouvoir s'arrêter (la nuit, un redémarrage, le jeu d'un autre) sans perdre ce qui a été fait. Dans forge, **Ctrl+C met en pause** : la boucle finit le pas en cours, sauvegarde tout (le modèle, l'état de l'optimiseur, l'endroit où l'on en est dans les données, et l'état du hasard), et s'arrête. Avec `--reprendre`, elle repart de cette sauvegarde.

Pour que la reprise soit **exacte**, il ne suffit pas de sauvegarder les poids : le prochain lot de texte et le prochain tirage du hasard doivent être ceux qui auraient suivi sans la pause.

`essai_pause_reprise` le vérifie : il entraîne deux fois le même modèle, une fois d'un trait, une fois arrêté au pas 30 (comme Ctrl+C) puis repris, et compare les deux pertes finales. Sur le processeur, elles sont identiques au bit près. Sur une carte graphique, deux entraînements identiques peuvent différer un peu, parce que la carte additionne dans un ordre qui varie : l'écart est alors minuscule, mais pas toujours nul.

In [ ]:
essai = essai_pause_reprise(appareil, racine)

print(f"D’un trait\u202f: perte {nombre(essai['perte_un_trait'], 6)} au pas {essai['pas_total']}.")
print(f"Arrêté au pas {essai['pas_de_la_pause']}, puis repris\u202f: perte {nombre(essai['perte_reprise'], 6)} au pas {essai['pas_total']}.")
print(f"Écart des deux pertes\u202f: {essai['ecart']}")
print(f"Les {essai['pas_total'] - essai['pas_de_la_pause']} pertes de la seconde moitié sont les mêmes une à une\u202f: {essai['pertes_identiques']}")
if appareil == "cpu":
    print(f"La reprise est exacte\u202f: {'oui' if essai['ecart'] == 0 else 'non'}")
else:
    print(f"Sur la carte, l’écart est petit\u202f: {'oui' if essai['ecart'] < 0.1 else 'non'} (il n’est pas forcément nul).")
print(f"Ce carnet a tourné sur {lieu}, en {nombre(essai['secondes'], 1)} s pour les trois entraînements.")

## Étape 5 : pour la version d'une heure

Tu viens de faire, en tout petit, ce que fait la version d'une heure de « ton IA » : le modèle « mini » (23,2 millions de paramètres), 6 370 pas de 131 072 jetons, 835 millions de jetons lus en une heure sur la carte graphique. Pour la refaire chez toi, il te faut :

- **une carte NVIDIA** : le processeur est environ 86 fois plus lent sur ce modèle (2 707 jetons par seconde contre 231 934). Une heure de carte serait environ 48 heures de processeur ;
- **le dépôt entier** (forge est dans `projets/forge`), et `uv`, qui installe PyTorch pour ta carte (`--extra gpu`) ;
- **de la place** : environ 1 Go de textes, 0,5 Go de jetons, et 0,6 Go pour les sauvegardes.

Les commandes, depuis `projets/forge`, dans l'ordre : télécharger les textes, les encoder, entraîner, évaluer.

```powershell
uv run --extra gpu python -m forge donnees --octets 1000M --dossier court
uv run --extra gpu python -m forge encoder --donnees court --tokenizer forge\ressources\tokenizer-ton-ia-v1.json
uv run --extra gpu python -m forge entrainer --config configs/mini.toml
uv run --extra gpu python -m forge evaluer --sauvegarde D:\forge-ia-donnees\entrainements\ton-ia-mini\sauvegardes\etape-0006370.pt --donnees court
```

Pendant l'entraînement, **Ctrl+C met en pause** (le pas en cours se termine, puis tout est sauvegardé). Pour reprendre exactement là où il s'est arrêté :

```powershell
uv run --extra gpu python -m forge entrainer --config configs/mini.toml --reprendre
```

Le journal de cette heure de calcul est dans le dépôt, résumé et réduit en courbe : `projets/1.5-pre-entrainement/mesures/ton-ia-mini.json`. La page du chapitre le montre.

In [ ]:
import json

releve_mini = json.loads((projet / "mesures" / "ton-ia-mini.json").read_text(encoding="utf-8"))
mini = releve_mini["resume"]
print(f"La version d’une heure, relevée le {releve_mini['releve_le']}\u202f:")
print(f"  {nombre(mini['pas_faits'])} pas sur {nombre(mini['pas_total'])}, {nombre(mini['jetons_vus'])} jetons vus, {nombre(mini['pauses'])} pause.")
print(f"  {nombre(mini['secondes_de_calcul'], 1)} s de calcul, {nombre(mini['debit_moyen'])} jetons par seconde.")
print(f"  perte de {nombre(mini['perte_premier_pas'], 3)} au premier pas à {nombre(mini['perte_dernier_pas'], 3)} au dernier, validation {nombre(mini['validation_finale'], 3)}.")

## Exercice : retrouver le pas de la pause

Quand un entraînement a été mis en pause puis repris, son journal le raconte : une ligne `{"evenement": "pause", "pas": 5}` marque la pause, suivie d'une ligne `reprise`. Écris la fonction `pas_de_la_pause(lignes)` qui rend le pas de la pause (ou `None` s'il n'y en a pas). `lignes` est la liste des lignes du journal, chacune un dictionnaire.

La cellule l'essaie sur le journal du dépôt (`tests/journal-factice.jsonl`) et sur un journal construit à part, et compare avec ce que dit `journal.sessions`. Si les deux réponses sont bonnes, le défi est réussi.

Indice : une ligne de pas n'a pas de clé `evenement`. Utilise `ligne.get("evenement")`.

In [ ]:
def pas_de_la_pause(lignes):
    # À toi : parcours les lignes, et rends le pas de la ligne dont l’évènement est « pause ».
    return None


# Ne change rien en dessous : cette partie essaie ta fonction sur deux journaux.
journal_du_depot = journal.lire(projet / "tests" / "journal-factice.jsonl")
journal_construit = (
    [{"evenement": "debut", "pas": 0}]
    + [{"pas": p, "perte": 10.0 - p} for p in range(1, 8)]
    + [{"evenement": "pause", "pas": 7}]
)
sans_pause = [{"evenement": "debut", "pas": 0}] + [{"pas": p, "perte": 10.0 - p} for p in range(1, 4)]
reussis = 0
for description, lignes_essai in (("le journal du dépôt", journal_du_depot), ("un journal construit", journal_construit), ("un journal sans pause", sans_pause)):
    voulu = next((s["dernier_pas"] for s in journal.sessions(lignes_essai) if s["fin"] == "pause"), None)
    obtenu = pas_de_la_pause(lignes_essai)
    if obtenu == voulu:
        reussis += 1
    else:
        print(f"Avec {description}, ta fonction rend {obtenu} au lieu de {voulu}.")
if reussis == 3:
    print("Défi réussi\u202f: ta fonction retrouve le pas de la pause dans les trois journaux.")
else:
    print(f"Pas encore ({reussis} essai sur 3).")

<details>
<summary>Voir une correction (essaie d'abord !)</summary>

Un journal est une liste de dictionnaires. Les lignes de pas n'ont pas de clé `evenement` : `ligne.get("evenement")` rend alors `None`, sans erreur. On cherche la première ligne dont l'évènement est `pause`, et on rend son pas. Sans pause, on arrive au bout de la boucle et on rend `None`.

Remplace la fonction de la cellule de l'exercice par :

```python
def pas_de_la_pause(lignes):
    for ligne in lignes:
        if ligne.get("evenement") == "pause":
            return ligne["pas"]
    return None
```

Un journal repris plusieurs fois a plusieurs pauses : cette fonction rend la première. `journal.sessions` les donne toutes, une par session.

</details>

In [ ]:
shutil.rmtree(racine, ignore_errors=True)  # le dossier temporaire de l’essai
print(f"Ce carnet a tourné sur {lieu}.")

## Pour aller plus loin

- Le code complet du projet, avec ses tests : [projets/1.5-pre-entrainement sur GitHub](https://github.com/chipat-neko/forge-ia/tree/main/projets/1.5-pre-entrainement).
- La page du chapitre, pour voir la courbe de la version d'une heure et ses sessions : [chapitre 1.5](https://chipat-neko.github.io/forge-ia/partie-1/1-5/).
- Depuis le dossier du projet, `uv run --extra cpu python -m pre_entrainement essai --appareil cpu` refait l'essai de pause et de reprise, et `uv run --extra cpu python -m pre_entrainement courbe --journal <ton journal> --config <ta configuration>` relève la courbe et les sessions d'un journal de forge.

Le code de ce carnet est sous licence MIT, ses textes sous licence CC BY 4.0.